# Laboratorio 2. El tablero no cuadra

Alguien dejó un pipeline que dice éxito. Finanzas dice que las ventas no cuadran con el archivo.

Su trabajo no es seguir una receta. Es dejar `oro_fact_ventas` usable y explicar qué apartaron.

Reglas del negocio, nada más:

- Una fila por `venta_id`.
- Solo entran las tiendas 1, 2 y 3.
- `monto` tiene que ser número. Si viene con coma, se interpreta. Si no se puede, se aparta.
- Una columna que no estaba en el contrato no entra al hecho.
- Lo apartado queda en `cuarentena_ventas`, con un motivo.
- Correr el notebook otra vez no duplica el hecho.

No editen el CSV. Si una prueba falla, el hallazgo es de ustedes.

In [0]:
# Carga directa del archivo de ventas desde tu Workspace
ruta = "file:/Workspace/Users/202401245@ujmd.edu.sv/ventas_export (1).csv"

bronce = spark.read.option("header", True).option("inferSchema", False).csv(ruta)
bronce.createOrReplaceTempView("bronce_ventas")

spark.sql("""
create or replace temp view dim_tienda as
select * from values
    (1, 'Centro'),
    (2, 'Escalon'),
    (3, 'Soyapango')
as t(tienda_id, nombre)
""")

print("Filas en el archivo:", bronce.count())
print("Columnas:", bronce.columns)

Filas en el archivo: 59
Columnas: ['venta_id', 'fecha', 'tienda_id', 'producto_id', 'cantidad', 'monto', 'medio_pago', 'canal']


## El pipeline que heredaron

Esta celda es el flujo roto. Léanla, no la copien como entrega. Carga todo, no aparta nada y si la corren dos veces duplica.

In [0]:
ruta = "file:/Workspace/Users/202401245@ujmd.edu.sv/ventas_export (1).csv"

bronce = spark.read.option("header", True).option("inferSchema", False).csv(ruta)
bronce.createOrReplaceTempView("bronce_ventas")

spark.sql("""
create or replace temp view dim_tienda as
select * from values
    (1, 'Centro'),
    (2, 'Escalon'),
    (3, 'Soyapango')
as t(tienda_id, nombre)
""")

print("Filas en el archivo:", bronce.count())
print("Columnas:", bronce.columns)

Filas en el archivo: 59
Columnas: ['venta_id', 'fecha', 'tienda_id', 'producto_id', 'cantidad', 'monto', 'medio_pago', 'canal']


## Su pipeline

Reemplacen esta celda. Tiene que crear `oro_fact_ventas` y `cuarentena_ventas`.

`oro_fact_ventas`: `venta_id`, `fecha`, `tienda_id`, `producto_id`, `cantidad`, `monto`, `medio_pago`.

`cuarentena_ventas`: las que no entran, con una columna `motivo`.

In [0]:
# 1. Preparación y limpieza inicial de datos
spark.sql("""
create or replace temp view bronce_preparado as
select 
    venta_id,
    fecha,
    tienda_id,
    producto_id,
    cantidad,
    monto as monto_orig,
    try_cast(replace(monto, ',', '') as double) as monto_clean,
    medio_pago,
    canal,
    row_number() over (partition by venta_id order by fecha desc) as rn
from bronce_ventas
""")

# 2. Creación de la tabla limpia (oro_fact_ventas)
spark.sql("""
create or replace temp view oro_fact_ventas as
select 
    venta_id,
    fecha,
    cast(tienda_id as int) as tienda_id,
    producto_id,
    cantidad,
    monto_clean as monto,
    medio_pago
from bronce_preparado
where rn = 1
  and try_cast(tienda_id as int) in (1, 2, 3)
  and monto_clean is not null
""")

# 3. Creación de la tabla de cuarentena (cuarentena_ventas)
spark.sql("""
create or replace temp view cuarentena_ventas as
select 
    venta_id,
    fecha,
    tienda_id,
    producto_id,
    cantidad,
    monto_orig as monto,
    medio_pago,
    canal,
    case 
        when rn > 1 then 'Venta ID duplicado'
        when try_cast(tienda_id as int) not in (1, 2, 3) or tienda_id is null then 'Tienda no permitida'
        when monto_clean is null then 'Monto invalido o no numérico'
        else 'Otro motivo'
    end as motivo
from bronce_preparado
where rn > 1
   or try_cast(tienda_id as int) not in (1, 2, 3)
   or tienda_id is null
   or monto_clean is null
""")


DataFrame[]

## Pruebas

No dicen cómo arreglarlo. Dicen si ya quedó. Las seis tienen que decir PASÓ.

In [0]:
oro = spark.table("oro_fact_ventas")
cuarentena = spark.table("cuarentena_ventas")
bronce = spark.table("bronce_ventas")

def prueba(nombre, ok):
    print(("PASÓ  " if ok else "FALLÓ ") + nombre)
    return ok

duplicados = spark.sql("""
select count(*) c from (
  select venta_id from oro_fact_ventas
  group by venta_id having count(*) > 1
)""").first()[0]

tiendas_malas = spark.sql("""
select count(*) c from oro_fact_ventas
where try_cast(tienda_id as int) not in (1, 2, 3)
""").first()[0]

montos_malos = spark.sql("""
select count(*) c from oro_fact_ventas
where try_cast(monto as double) is null
""").first()[0]

columnas_de_mas = [c for c in oro.columns if c.lower() == "canal"]
sin_motivo = "motivo" not in [c.lower() for c in cuarentena.columns]
oro_inflado = oro.count() >= bronce.count()
cuarentena_vacia = cuarentena.count() == 0

ok = []
ok.append(prueba("una fila por venta_id", duplicados == 0))
ok.append(prueba("solo tiendas 1, 2 y 3", tiendas_malas == 0))
ok.append(prueba("monto numérico en el hecho", montos_malos == 0))
ok.append(prueba("canal no entra al hecho", len(columnas_de_mas) == 0))
ok.append(prueba("cuarentena con motivo y con filas", (not sin_motivo) and (not cuarentena_vacia)))
ok.append(prueba("oro menor que el archivo", not oro_inflado))
print("-")
print("LISTO" if all(ok) else "SIGAN. Hay pruebas en rojo.")

PASÓ  una fila por venta_id
PASÓ  solo tiendas 1, 2 y 3
PASÓ  monto numérico en el hecho
PASÓ  canal no entra al hecho
PASÓ  cuarentena con motivo y con filas
PASÓ  oro menor que el archivo
-
LISTO


## Desafío o reto

Cuando las seis digan PASÓ, respondan en esta celda, sin mirar la guía:

1. ¿Qué pasó con la misma venta llegada dos veces? 
Se identificó usando ROW_NUMBER() OVER (PARTITION BY venta_id ORDER BY fecha DESC). La venta más reciente se envió a oro_fact_ventas (rn = 1) y la duplicada a cuarentena_ventas (rn > 1).
2. ¿Qué hicieron con un monto que venía en texto? 
Se le quitó la coma con REPLACE() y se convirtió a decimal usando TRY_CAST(). Si el monto era inválido o NULL, se mandó a cuarentena_ventas.
3. ¿Por qué una tienda se quedó fuera? 
Porque solo se permitieron las tiendas 1, 2 y 3 segun las reglas del negocio. Cualquier otro valor de tienda_id se desvió a cuarentena como "Tienda no permitida".
4. ¿Dónde quedó la columna que no estaba en el contrato?
La columna canal no se incluyó en oro_fact_ventas para respetar el esquema esperado, pero se conservó en cuarentena_ventas para trazabilidad y auditoría.
5. Si corren el notebook otra vez, ¿el hecho crece?
No, porque el pipeline crea vistas temporales (CREATE OR REPLACE TEMP VIEW) que se sobrescriben en cada ejecución, garantizando que el proceso sea idempotente.

El que solo pega la consulta y no puede explicar una de estas, no termino el laboratorio